In [ ]:
import hail as hl
import os

In [ ]:
from ukb_utils import hail_init
from ukb_utils import genotypes
from ko_utils import qc
from ko_utils import analysis

In [ ]:
os.chdir('/well/lindgren/UKBIOBANK/flassen/projects/KO/wes_ko_ukbb/')

In [4]:
hail_init.hail_bmrc_init('logs/hail/hail_debug_test.log', 'GRCh38')

Running on Apache Spark version 3.1.2
SparkUI available at http://compe065.hpc.in.bmrc.ox.ac.uk:4040
Welcome to
     __  __     <>__
    / /_/ /__  __/ /
   / __  / _ `/ / /
  /_/ /_/\_,_/_/_/   version 0.2.74-0c3a74d12093
LOGGING: writing to logs/hail/hail_debug_test.log


In [39]:
#
mt1 = qc.get_table('data/mt/ukb_wes_200k_annotated_chr22.mt', 'mt')
#mt1 = mt1.annotate_rows(vep = mt1.vep.annotate(Gene = mt1.vep.worst_csq_by_gene_canonical.gene_id))
#mt1 = qc.filter_min_missing(mt1, 0.05)
mt1 = qc.filter_max_maf(mt1, 0.02)
mt1 = analysis.filter_vep(mt1, 'consequence_category', ['ptv','damaging_missense'])
mt1.count()

(0, 199795)

In [ ]:
    
    
in_mt = in_mt.explode_rows(in_mt.vep.worst_csq_by_gene_canonical)
    # annotate with rsid
    in_mt = in_mt.annotate_entries(rsid_entry = in_mt.rsid)
    in_mt = in_mt.annotate_entries(snpid_entry = in_mt.snpid)
    in_mt = in_mt.annotate_entries(af_entry = in_mt.info.AF)
    in_mt = in_mt.annotate_entries(ac_entry = in_mt.info.AC)
    in_mt = in_mt.annotate_entries(rsid_gt = hl.delimit([
        hl.str(in_mt.GT),
        #in_mt.snpid_entry,
        in_mt.rsid_entry], ';'))

In [25]:
mt2 = qc.get_table('data/mt/ukb_wes_200k_annotated_chr22_singletons.mt', 'mt')
mt2 = mt2.annotate_rows(vep = mt2.vep.annotate(Gene = mt2.vep.worst_csq_by_gene_canonical.gene_id))
mt2 = qc.filter_min_missing(mt2, 0.05)
mt2 = analysis.filter_vep(mt2, 'consequence_category', 'ptv')

In [30]:
def gene_csqs_calc_pKO(mt_phased, mt_unphased, fields_drop = ['DT','singletons']):
    '''Annotates entries with P(Knockout). Requires a phased matrix and an unphased matrix that only contains singletons.'''
    
    # setup variables
    mt1 = mt_phased # contains phased non-singletons
    mt2 = mt_unphased # contains unphased singletons
    
    # Determine probability of being KO given singletons and phased hetz
    mt1_burden = gene_csqs_dosage_builder(mt1)
    mt2_burden = gene_burden_annotations_per_sample(mt2)
    mt_ko = mt1_burden.annotate_entries(singletons = mt2_burden[(mt1_burden.Gene, mt1_burden.s)].n)
    mt_ko = mt_ko.annotate_entries(singletons = hl.if_else(~hl.is_missing(mt_ko.singletons), mt_ko.singletons, 0 ))
    mt_ko = calc_p_ko(mt_ko)

    # drop not needed rows
    if fields_drop is not None:
        mt_ko = mt_ko.drop(*[f for f in fields_drop if f in mt_ko.entry])
    return mt_ko

In [33]:
#gene_csqs_calc_pKO(mt1, mt2, fields_drop = None).describe()
res = gene_csqs_calc_pKO(mt1, mt2, fields_drop = None)
res = res.entries()

In [34]:
res.filter(res.pKO > 0.9).show()

,,,,,
Gene,s,eur,DT,singletons,pKO
array<str>,str,int32,int32,int64,float64
